# Cavity number preprocessing
Students should develop a software program to preprocess an image and get it ready to perform the
OCR of the cavity number of a plastic cap.
The cap has an external tab at a fixed position in relation to the cavity number.

## Task 0: Imports and setup

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import sys

# Only for jupyter notebook visualization
%matplotlib inline 

filename="./img/g_09.bmp"
image = cv2.imread(filename, cv2.COLOR_BGR2GRAY)
# image = cv2.resize(image, (0, 0), fx =2, fy = 2)
plt.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.show()

## Task 1: Generate a crop of the cavity number
### 1.1: Outline the cap by generating a circle that fits the cap mouth

#### 4) WORKING SOLUTION Delete objects around 

In [ ]:
_, image_bin = cv2.threshold(image.astype(np.uint8), 20, 255, cv2.THRESH_BINARY)
edge_detected_image = cv2.Canny(image_bin, 75, 100)

middle_line = edge_detected_image[:, int(edge_detected_image.shape[1]/2)]
non_zero_indices = np.nonzero(middle_line)[0]
diameter = non_zero_indices[-1]-non_zero_indices[0]
radius = int(diameter/2)

edge_detected_image[:, 0:int(edge_detected_image.shape[1]/2) - int(radius*1.05)] = 0
edge_detected_image[:, int(edge_detected_image.shape[1]/2) + int(radius*1.05):] = 0

edge_detected_image[0:int(edge_detected_image.shape[0]/2) - int(radius*1.1), :] = 0
edge_detected_image[int(edge_detected_image.shape[0]/2) + int(radius*1.1):, :] = 0

rows = image.shape[0]

found = False
par2 = 25
while not found and par2 > 5:
    circles = cv2.HoughCircles(edge_detected_image, 
                            cv2.HOUGH_GRADIENT, # Detection method unique that is implemented
                            1, 
                            rows/4,
                            param1=500, # Theshold on the gradient
                            param2=par2)  # Smaller find more false positives
    if circles is not None and len(circles) > 0:
        found = not found
    par2 -= 5

ax = plt.subplot(1,1,1)
ax.set_title(filename)
if circles is not None:
    if len(circles[0]) > 1:
        print(f"Image {filename} has 2 or more circles")
    for circle in circles[0]:
        center = [circle[0], circle[1]]
        radius = circle[2]
        center_in_pixels = np.int16(np.around(center))
        radius_in_pixels = np.int16(np.around(radius))
        circle = plt.Circle(center, radius, color='m', fill=False)
        ax.add_patch(circle)
else:
    print(f"Image {filename} has no circles") 

ax.imshow(image, cmap='gray', vmin=0, vmax=255)

plt.show()

### 1.2 Generate a crop containing the cavity number. The crop should contain the cavity number and it should appear upright

In [ ]:
radius_inside = int(radius)
radius_outside = radius_inside + 15

fig,ax = plt.subplots(1)
circle_inside = plt.Circle(center, radius_inside, color='m', fill=False)
circle_ouside = plt.Circle(center, radius_outside, color='b', fill=False)
ax.add_patch(circle_inside)
ax.add_patch(circle_ouside)
ax.imshow(image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
# Create a mask with the same dimensions as the image
mask = np.zeros(image_bin.shape[:2], dtype=np.uint8)

# Draw a transparent circle on the mask
cv2.circle(mask, center_in_pixels, radius_outside, (255), thickness=-1)
# Draw a black circle on the mask
cv2.circle(mask, center_in_pixels, radius_inside, (0), thickness=-1)

# Extract the circular ROI using the mask
circular_roi = cv2.bitwise_and(image_bin, image_bin, mask=mask)

# Filter to delete some white areas that are not the tab
# an alternative to median filtering could be to remove those conn. components 
# that do not exceed a certain size, they could be e.g. spurious nonblack pixels 
# that may be an obstacle for further elaborations
# circular_roi_median = circular_roi 
# circular_roi_median = cv2.medianBlur(circular_roi, 3)
# circular_roi_median = cv2.morphologyEx(circular_roi_median, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)), iterations=3)
# circular_roi_erode = cv2.erode(circular_roi, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)), iterations=2)
# circular_roi_median = cv2.erode(circular_roi, np.ones((3, 3), np.uint8), iterations=3)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)) # The dimension of the kernel is not scale invariant (if you make the image smaller)
circular_roi_opened = cv2.morphologyEx(circular_roi, cv2.MORPH_OPEN, kernel, iterations=2)

# Display the original image and the circular ROI
fig,ax = plt.subplots(1)
circular_roi_rgb = cv2.cvtColor(circular_roi_opened, cv2.COLOR_BGR2RGB)
ax.imshow(circular_roi_rgb)
plt.show()

In [ ]:
num_labels, labels_im, stats, centroids = cv2.connectedComponentsWithStats(circular_roi_opened)

if len(centroids) > 2:
    print(f"Found {len(centroids) - 1} object in the anular region, they are too much")

    WIDTH = 2
    HEIGHT = 3
    AREA = 4
    best_i = -1
    best_rect = 0

    # The residual object are the tab and some arc region that is too big to be removed with the opening
    # We can find the tab looking at the most rectangular object 
    for i, stat in enumerate(stats[1:]): # Altrenativa a median: [x for x in stats[1:] if x[AREA] > 20]
        print(stat)
        rect = stat[AREA] / (stat[WIDTH] * stat[HEIGHT])
        if rect > best_rect:
            best_rect = rect
            best_i = i+1 # we start from one to skip the background
    centroids = [centroids[0], centroids[best_i]]
elif len(centroids) < 2:
    print("Tab not found!")
    sys.exit(1)

# 0 is the backgorud, 1 is the tab
tab_center = centroids[1]

def imshow_components(labels):
    # Map component labels to hue val
    label_hue = np.uint8(179*labels/np.max(labels))
    blank_ch = 255*np.ones_like(label_hue)
    labeled_img = cv2.merge([label_hue, blank_ch, blank_ch])

    # cvt to BGR for display
    labeled_img = cv2.cvtColor(labeled_img, cv2.COLOR_HSV2BGR)

    # set bg label to black
    labeled_img[label_hue==0] = 0

    # center 
    center_patch = plt.Circle(tab_center, 1, color="g")


    _ ,ax = plt.subplots(1)
    ax.imshow(labeled_img)
    ax.add_patch(center_patch)
    plt.show()

imshow_components(labels_im)

In [ ]:
tab_center = np.int16(np.around(tab_center))
center = np.int16(np.around(center))

# Slope of the straight line that connects the center of the tab with the center of the cap
m = (center[1] - tab_center[1]) / (center[0] - tab_center[0]) * 1.0
# Intercept
q = center[1] - m * center[0]

line = np.polynomial.polynomial.polyline(q, m) # idk why but m is always the opposite of what expected

x_val = ([*range(tab_center[0],center[0])] , [*range(center[0], tab_center[0])])[int(tab_center[0] > center[0])]
y_val = np.polynomial.polynomial.polyval(x_val, line)

fig,ax = plt.subplots(1)
ax.imshow(image, cmap='gray', vmin=0, vmax=255)
ax.plot(x_val, y_val, color="b")
ax.plot(np.ones(image.shape[0]-1)*center[0],[*range(1, image.shape[0])], color="g")
plt.show()

In [ ]:
# Find the angle between the green and the blue lines
x = np.abs(tab_center[0]-center[0])
y = np.abs(tab_center[1]-center[1])

teta = np.arctan(x/y)

# Form rad to deg
rotation = (teta * 180 / np.pi)

# Looking at the slope we choose the direction of the rotation
if m > 0 :
    rotation = rotation * -1

# Looking at the tab position we choose if we have to overturn
if tab_center[1] > image.shape[0] / 2:
    rotation += 180


image_center = tuple(np.array(image.shape[1::-1]) / 2)
rot_mat = cv2.getRotationMatrix2D(image_center, rotation, 1.0)
image_with_vertical_tab = cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)
fig,ax = plt.subplots(1)
ax.imshow(image_with_vertical_tab, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
cavity_numer_crop = image_with_vertical_tab.copy()[100:170, 310:450]
plt.imshow(cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()

#### More robust cropping

In [ ]:
print(center)
cavity_numer_crop = image_with_vertical_tab.copy()[center[1] - int(radius*0.8) : center[1] - int(radius*0.5), center[0] - int(radius*0.4) : center[0] + int(radius*0.4)]
plt.imshow(cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()

## 2 Apply a polar transform

In [ ]:
flags = cv2.INTER_CUBIC | cv2.WARP_FILL_OUTLIERS | cv2.WARP_POLAR_LINEAR
polar_image = cv2.warpPolar(image_with_vertical_tab, image_with_vertical_tab.shape, image_center, 360, flags)
fig,ax = plt.subplots(1)
ax.imshow(polar_image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
rotated_polar_image = cv2.rotate(polar_image, cv2.ROTATE_90_COUNTERCLOCKWISE)
fig,ax = plt.subplots(1)
ax.imshow(rotated_polar_image, cmap='gray', vmin=0, vmax=255)
plt.show()

In [ ]:
rect_cavity_numer_crop = rotated_polar_image.copy()[300:360, 520:640]
cv2.imwrite("result.png", rect_cavity_numer_crop)
plt.imshow(rect_cavity_numer_crop, cmap='gray', vmin=0, vmax=255)
plt.show()

## 4: Not required but apply ocr

In [ ]:
from PIL import Image
import pytesseract
pytesseract.image_to_string(
    Image.open('/Users/micheletagliani/Developer/Python/CoputerVision/CvCavityNumberPreprocessing/result.png'),
    config="--psm 7")